---

<big><big> **Project 2** </big></big>

**Course:** CSCE-636-700

**Instructor:** Dr. Anxiao Jiang

**Author:** Brody Jordan (431005357)

**Due Date:** November 14, 2025 @ 11:59 PM CST

---

# **Training Data**



## **Provided Data**

We were provided a new set of training data (with old set as a subset) that contains 44,399 samples in a list of the form [n, k, m, P], where $n=9$, $k \in \{4,5,6\}$, and $m \in \{2,3,\dots,n-k\}$. The list of possible combinations of $n$, $k$, and $m$ is shown here:

 1. $~n=9$, $k=4$, $m=2$.
 2. $~n=9$, $k=4$, $m=3$.
 3. $~n=9$, $k=4$, $m=4$.
 4. $~n=9$, $k=4$, $m=5$.
 5. $~n=9$, $k=5$, $m=2$.
 6. $~n=9$, $k=5$, $m=3$.
 7. $~n=9$, $k=5$, $m=4$.
 8. $~n=9$, $k=6$, $m=2$.
 9. $~n=9$, $k=6$, $m=3$.

Downloading that data to the runtime instance:



In [3]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

In [ ]:
!pip install gdown tabulate keras keras-hub scikit-learn

In [4]:
# Load the data using gdown (was having issues mounting my drive)
import gdown

data_id = "1BD_oGoFLWB2VXwyWHS9zWYp_LXpcE847"
targets_id = "1Cs5nQZNfji_1SfDKP5MhIDmO4kag9enm"

gdown.download(id=data_id, output="training_data", quiet=True)
gdown.download(id=targets_id, output="training_targets", quiet=True)

'training_targets'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Loading (via pickle) and validating data loading by printing a random training element:

In [5]:
# Loading the downloaded training data and targets
import pickle
import random

with open("training_data", "rb") as f:
    raw_data = pickle.load(f)

with open("training_targets", "rb") as f:
    raw_heights = pickle.load(f)

# Print random training data element
r_idx = random.randint(0, len(raw_data) - 1)
n, k, m, P = raw_data[r_idx]
height = raw_heights[r_idx]
print(f"""
Random element (idx: {r_idx}) of training dataset:
n = {n}, k = {k}, m = {m},
P = {P}, \n\n m-height = {height}
""")


Random element (idx: 10595) of training dataset:
n = 9, k = 6, m = 2,
P = [[ 14. -14.  14.]
 [  8.   5.  -5.]
 [  4.  11.   4.]
 [  7.   0.  18.]
 [ 10. -12. -10.]
 [  8.  15.   7.]], 

 m-height = 107.92307692307693



Training data elements are in the shape [n, k, m, [P]]

Here I sort the data into a dictionary keyed by tuple $(n,k,m)$. So performing the lookup `nkm_data[(9, 4, 5)]` gives a list of all $P$ matrices with $(n,k,m)=(9,4,5)$, and similarly for indexing the `nkm_heights` dictionary.

Looking at how many samples each possible [n,k,m] pair has:

In [6]:
import numpy as np
from collections import defaultdict
from tabulate import tabulate

data_dict = defaultdict(list)
heights_dict = defaultdict(list)

for sample, height in zip(raw_data, raw_heights):
    nkm_tuple = tuple(sample[:3])
    data_dict[nkm_tuple].append(np.array(sample[3]))
    heights_dict[nkm_tuple].append(height)

table_data = [(n, k, m, len(P)) for (n, k, m), P in data_dict.items()]
print(tabulate(table_data, headers=["n", "k", "m", "Samples (num P)"], tablefmt="outline"))

+-----+-----+-----+-------------------+
|   n |   k |   m |   Samples (num P) |
+=====+=====+=====+===================+
|   9 |   6 |   2 |             11546 |
|   9 |   4 |   2 |              4109 |
|   9 |   6 |   3 |              6147 |
|   9 |   5 |   4 |              2887 |
|   9 |   4 |   4 |              3689 |
|   9 |   4 |   3 |              4007 |
|   9 |   5 |   2 |              5007 |
|   9 |   5 |   3 |              4633 |
|   9 |   4 |   5 |              2374 |
+-----+-----+-----+-------------------+


## Data Preprocessing Functions



Since I will be training 9 different models, I have this function `get_data()` to retrieve data from by $(n, k, m)$.


You pass $(n, k, m)$ into `get_data()`, and it returns a tuple containing a list of all flattened matrices $P$ for $(n, k, m)$ and a list of all corresponding true heights for $(n, k, m)$.

In [22]:
def get_flat_nkm_data(n, k, m):
    nkm_data = np.array(data_dict[(n, k, m)], dtype=np.float32)

    # UPDATED: using CNN now, flattening not needed...
    # nkm_data = nkm_data.reshape(len(nkm_data), k*(n-k))

    nkm_heights = np.array(heights_dict[(n, k, m)], dtype=np.float32)

    return nkm_data, nkm_heights

Here I have a function to split the data into training, validation, and test sets. You pass some unseparated dataset into `split_data()` and it returns

In [39]:
from sklearn.model_selection import train_test_split as split

def split_and_normalize_data(data, heights, nkm):

    # Split data into training and test data
    X_train_, X_test, y_train_, y_test = split(data, heights, test_size=0.15)
    # Split training data into training and validation data
    X_train, X_val, y_train, y_val = split(X_train_, y_train_, test_size=0.10)

    # Normalize train, test, and validation data by mean and std of training data
    # MOVED TO NORMALIZATION LAYER AT MODEL INPUT...

    return X_train, y_train, X_val, y_val, X_test, y_test

## **Data Generation from LP**

An expanded training dataset would be valuable in reducing overfitting and improving model accuracy. It would also likely improve the loss for the $(n,k,m)=(9,4,5)$ sets, as they only have $1494$ training samples (as shown above).

If I have time, I will implement the LP here.

In [ ]:
# LP code to generate additional data

---

# **Model Design**

**Model inputs**
1. $n$ (integer)
2. $k$ (integer)
3. $m$ (integer)
4. $P$ ($k \times (n-k)$ matrix)

**Model outputs**
1. Prediction of the $m$-height, $h \in \{x \in \mathbb{R} \mid x \geq 1\}$

With the training data prepared, it is time to design the structure of the model.

We were also provided some target values of $n$, $k$, and $m$ to focus on:

\begin{equation}
    n = 9, \quad k \in \{4,5,6\}, \quad m \in \{2,3,\dots,n-k\}
\end{equation}

So that is what I will do.

## Model loss definition

Here I defined a custom loss function `loss2_squared_error()`.

It computes $\sigma(y, \hat{y}) \triangleq \left(\log_2 y - \log_2 \hat{y}\right)^2$ where $y$ is the true height and $\hat{y}$ is the predicted height. I adapted this function from the Keras mean-square-logarithmic-error loss function here: [Keras GitHub repo](https://github.com/keras-team/keras/blob/v3.3.3/keras/src/losses/losses.py#L1255-L1288).

That way my model is minimizing the cost at the end of Chapter 4.


In [159]:
from keras import ops, backend

# Define our custom loss function (adapted from MLSE)
def loss2_squared_error(y_true, y_pred):
    epsilon = ops.convert_to_tensor(backend.epsilon())
    y_pred = ops.convert_to_tensor(y_pred)
    y_true = ops.convert_to_tensor(y_true, dtype=y_pred.dtype)
    first_log = ops.log2(ops.maximum(y_pred, epsilon))
    second_log = ops.log2(ops.maximum(y_true, epsilon))
    return ops.mean(ops.square(first_log - second_log), axis=-1)

## Model definition

Here I also define a `train_model()` function that takes some input $(n,k,m)$ and some training and validation data (and corresponding heights). It returns a compiled sequential keras model.

For this updated model, I am attempting to use a CNN based architecture, rather than just flattening each matrix and using a standard DNN.

I have a normalization layer near the input, with a 2D convolution layer (32, (3, 3)) $\to$ 2D max pooling (2, 2) $\to$ 2D conv. (64, (3, 3)) $\to$ dense (64) $\to$ dropout (0.2) $\to$ dense (1).

The minimum of ReLU is 0, so I put a lambda layer at the end so I can force the output to be >= 1.

In [160]:
import keras
from keras import layers, optimizers
from keras.regularizers import l1, l2

def create_model(n, k, m):

    # norm = layers.Normalization()
    # norm.adapt(X_train.astype("float32"))

    model = keras.Sequential([
        layers.Input(shape=(k, n-k, 1)),
        # norm,
        layers.Normalization(name="norm_layer"),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", kernel_regularizer=l2(1e-4)),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same", kernel_regularizer=l2(1e-4)),

        layers.GlobalAveragePooling2D(),

        layers.Dense(64, activation="relu", kernel_regularizer=l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(1, activation="relu"),
        layers.Lambda(lambda x: keras.ops.add(x, 1.0))
    ])

    learning_rate = 3e-4

    model.compile(
        optimizer=optimizers.AdamW(learning_rate=learning_rate),
        loss=loss2_squared_error,
        metrics=["mae"]
    )

    return model

---

# **Model Training**

Time to train the models... O_O

Here the models run for 500 epochs with a batch size of 64. THey are saved by name as .keras files.

In [166]:
import keras
import numpy as np

def train_model(n, k, m, X_train, y_train, X_val, y_val):

    print(f"Training model for (n, k, m) = ({n}, {k}, {m})...")

    model = create_model(n, k, m)

    # print(model.summary())

    norm_layer = model.get_layer("norm_layer")
    norm_layer.adapt(X_train[..., np.newaxis])

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=f"{n}_{k}_{m}_trained_model.keras",
            save_best_only=True,
            monitor="val_loss",
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            restore_best_weights=True,
            patience=20,
            verbose=1,

        )
    ]

    history = model.fit(
        X_train,
        y_train,
        epochs=200,
        batch_size=32,
        validation_data=(X_val[..., np.newaxis], y_val),
        callbacks=callbacks,
        verbose=0,
    )

    print(f"Model fitting for (n, k, m) = ({n}, {k}, {m}) completed. Saved to ./{n}_{k}_{m}_trained_model.keras")

    return model, history

## Training each model

Loop over each $(n,k,m)$. For each one of the 9 pairs, collect the data, normalize the data and split it into training, validation, and testing data. Then train the model with that data and calculate the avg cost.

In [167]:
import numpy as np

def calculate_avg_cost(model, X_test, y_test):
    predicted_height = model.predict(X_test).reshape(-1)
    cost = (np.log2(y_test) - np.log2(predicted_height))**2
    avg_cost = np.mean(cost)
    return avg_cost

all_nkm_avg_costs = []

for nkm in data_dict.keys():

    # Get data for (n, k, m)
    data, heights = get_flat_nkm_data(*nkm)
    data_train, heights_train, data_val, heights_val, data_test, heights_test  = split_and_normalize_data(data, heights, nkm)
    print(f"For {nkm}: {len(data_train)} training datapoints, {len(data_val)} validation datapoints, {len(data_test)} test datapoints.")

    # Create and train model
    model, history = train_model(*nkm, data_train, heights_train, data_val, heights_val)
    avg_cost = calculate_avg_cost(model, data_test[..., np.newaxis], heights_test)

    print(f"Average cost for {nkm} testset: {avg_cost}\n{'-'*75}\n")
    all_nkm_avg_costs.append(avg_cost)

print(f"Average cost across all (n, k, m): {np.mean(all_nkm_avg_costs)}")
# assert(len(all_nkm_avg_costs) == 9)



For (9, 6, 2): 8832 training datapoints, 982 validation datapoints, 1732 test datapoints.
Training model for (n, k, m) = (9, 6, 2)...
Restoring model weights from the end of the best epoch: 200.
Model fitting for (n, k, m) = (9, 6, 2) completed. Saved to ./9_6_2_trained_model.keras
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
Average cost for (9, 6, 2) testset: 0.5188249945640564
---------------------------------------------------------------------------

For (9, 4, 2): 3142 training datapoints, 350 validation datapoints, 617 test datapoints.
Training model for (n, k, m) = (9, 4, 2)...
Restoring model weights from the end of the best epoch: 200.
Model fitting for (n, k, m) = (9, 4, 2) completed. Saved to ./9_4_2_trained_model.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step
Average cost for (9, 4, 2) testset: 0.4785168170928955
---------------------------------------------------------------------------

For (9, 6, 3): 4701 training datapoints, 523 validation datapoints, 923 test datapoints.
Tr

## Calculating m-heights for input data


In order for this function to run, the models need to be in the runtime environment. You will probably(?) need to upload them to the runtime. They need to be in the ./ directory.

In [179]:
import numpy as np
import keras
from keras import layers, ops, backend
from keras.saving import register_keras_serializable
import builtins

keras.config.enable_unsafe_deserialization()

@register_keras_serializable(package="custom", name="loss2_squared_error")
def loss2_squared_error(y_true, y_pred):
    epsilon = ops.convert_to_tensor(backend.epsilon())
    y_pred = ops.convert_to_tensor(y_pred)
    y_true = ops.convert_to_tensor(y_true, dtype=y_pred.dtype)
    first_log = ops.log2(ops.maximum(y_pred, epsilon))
    second_log = ops.log2(ops.maximum(y_true, epsilon))
    return ops.mean(ops.square(first_log - second_log), axis=-1)

# hacky workaround
builtins.keras = keras

def get_predictions(raw_data):
    predictions = []

    # load models
    loaded_models = {}

    # loop over raw_data list and make predictions
    for datapoint in raw_data:
        nkm = (tuple(datapoint[:3]))
        P = datapoint[3]
        k, n_k = P.shape
        P = np.reshape(P, (1, k, n_k, 1))

        target_model = f"{nkm[0]}_{nkm[1]}_{nkm[2]}_trained_model.keras"

        if target_model not in loaded_models:
            loaded_models[target_model] = keras.models.load_model(target_model)
            print(f"loaded model: {target_model}")

        model = loaded_models[target_model]

        prediction = model.predict(P.astype("float32"), verbose=0)
        predictions.append(prediction)

    return [float(pred.squeeze()) for pred in predictions]

predictions = get_predictions(raw_data)


loaded model: 9_6_2_trained_model.keras
loaded model: 9_4_2_trained_model.keras
loaded model: 9_6_3_trained_model.keras
loaded model: 9_5_4_trained_model.keras
loaded model: 9_4_4_trained_model.keras
loaded model: 9_4_3_trained_model.keras
loaded model: 9_5_2_trained_model.keras
loaded model: 9_5_3_trained_model.keras
loaded model: 9_4_5_trained_model.keras


Dumping my predicted heights to file...

In [178]:
with open("final_predicted_heights", "wb") as f:
    pickle.dump(predictions, f)


---

# **Running the model**

Here I have provided a way for you to interface with the model fairly easily.

In [172]:
# Load your data here :D


# I will assume it is a tuple of the form new_data = (n, k, m, [P]) (exactly the same as the input file provided)
predictions = get_predictions(new_data) # <- you place new_data :)



NameError: name 'new_data' is not defined

Thank you!!!